In [1]:
# --- Rutas del proyecto -------------------------------------------------
# Definición única en projects/mayanlab/mayanlab/paths.py. Funciona desde
# cualquier directorio de trabajo: no hay rutas relativas en este notebook.
from pathlib import Path

from mayanlab.paths import (
    AUDIO, ANNOTATIONS, DATA, MANIFESTS, SEGMENTS, TRANSCRIPTS, WORK,
    NARRACIONES_MP3, NARRACIONES_TRANSCRIPTS, CLEAN_TEXT, ensure, project,
)

P = project("corpus")
FIGURES = P.figures
ensure(FIGURES)

In [2]:
from pathlib import Path

def get_text_file(path):
    with open(path, "r") as f:
        return f.read()

In [3]:
import glob

# Los 15 hablantes del 3h viven ya en un único sitio y con el mismo formato
# (un bloque por utterance separado por línea en blanco). Antes 01-03 había que
# sacarlos aparte de la columna `correction` del CSV; ahora no.
text_files = sorted(glob.glob(str(TRANSCRIPTS / "*.txt")))
text_files

['/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/01_Anatolio_Pech.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/02_Liboria_May.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/03_Eligio_Uicab_Jatswooj.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/04_Alfonso_Tamay.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/05_Felipe_May.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/06_Gricelda_Pech.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/07_Eligio_Uicab_Tomojchi.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/08_Teodoro_May.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/09_Adolfo_Chuc.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/10_Jesus_Euan.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/data/final/transcripts/11_Hector_May.txt',
 '/home/maucr/Documentos/thesis-mayan-ai/d

In [4]:
import pandas as pd

# Comprobación: los bloques de cada .txt deben cuadrar con los segmentos anotados.
expected = pd.read_csv(ANNOTATIONS / "3h-transcripts.csv")["audio_file"].value_counts()

for file in text_files:
    stem = Path(file).stem
    n = len(get_text_file(file).split("\n\n"))
    print(f"{stem:<28} {n:>4} utterances  (esperadas: {expected.get(stem, 0)})")

print("\ntotal:", sum(len(get_text_file(f).split("\n\n")) for f in text_files),
      "| esperadas:", int(expected.sum()))

01_Anatolio_Pech               39 utterances  (esperadas: 39)
02_Liboria_May                 38 utterances  (esperadas: 38)
03_Eligio_Uicab_Jatswooj       47 utterances  (esperadas: 47)
04_Alfonso_Tamay               66 utterances  (esperadas: 66)
05_Felipe_May                  81 utterances  (esperadas: 81)
06_Gricelda_Pech               43 utterances  (esperadas: 43)
07_Eligio_Uicab_Tomojchi       50 utterances  (esperadas: 50)
08_Teodoro_May                123 utterances  (esperadas: 123)
09_Adolfo_Chuc                 64 utterances  (esperadas: 64)
10_Jesus_Euan                  60 utterances  (esperadas: 60)
11_Hector_May                  31 utterances  (esperadas: 31)
12_Lourdes_y_Marcela_Ucam      56 utterances  (esperadas: 56)
13_Venustiano_Puc             140 utterances  (esperadas: 140)
14_Micaela_Ek                 162 utterances  (esperadas: 162)
15_Mario_Chan                 227 utterances  (esperadas: 227)

total: 1227 | esperadas: 1227


In [5]:
spk_metadata = pd.read_csv(ANNOTATIONS / "spk_metadata.csv")
spk_metadata[spk_metadata[" description"].str.contains("Eligio")].iloc[0]

def get_spk_id(row):
    if row["spk_id"].startswith("spk"):
        return row["spk_id"]

    name = row["spk_id"][3:10]
    info_spk = spk_metadata[spk_metadata[" description"].str.contains(name)].iloc[0]
    new_spk_id = info_spk["spk_id"]

    return new_spk_id

In [6]:
utterances = []

for file in text_files:
    text_list = get_text_file(file).split("\n\n")
    file_name = Path(file).stem

    for idx, texto in enumerate(text_list):
        utterances.append({
            "utt_id": f"{file_name}_{idx+1}",
            "maya": texto.strip(),
            "spk_id": file_name,
        })

len(utterances)

1227

In [7]:
df_3h = pd.DataFrame(utterances)

In [8]:
df_3h.head(5)


,utt_id,maya,spk_id
0,01_Anatolio_Pech_1,xtakumbil xunáan le tsikbalo u k'aaba'e xta'ak...,01_Anatolio_Pech
1,01_Anatolio_Pech_2,jaaj ta'akumbil xunáano le xunáano jump'éel ko...,01_Anatolio_Pech
2,01_Anatolio_Pech_3,u k'aaba'e ya'axche palomeke u ya'ala'al ti u ...,01_Anatolio_Pech
3,01_Anatolio_Pech_4,káan u yila tu bin tu taal u yáa'biltale ku yo...,01_Anatolio_Pech
4,01_Anatolio_Pech_5,pero leti'e u tuukul beyo le káan weenek le u ...,01_Anatolio_Pech


In [9]:
df_3h["new_spk_id"] = df_3h.apply(lambda x: get_spk_id(x), axis=1)

df_3h[df_3h["new_spk_id"] == "spk_021"]

,utt_id,maya,spk_id,new_spk_id
77,03_Eligio_Uicab_Jatswooj_1,le wa a k'áat ka in tsikbalt teech maaya leti'...,03_Eligio_Uicab_Jatswooj,spk_021
78,03_Eligio_Uicab_Jatswooj_2,ja'asajáóol je'elo ku jook'olo'ob k'áaxo yaan ...,03_Eligio_Uicab_Jatswooj,spk_021
79,03_Eligio_Uicab_Jatswooj_3,ku báalantikuba'ob tu paach le che xano utia'a...,03_Eligio_Uicab_Jatswooj,spk_021
80,03_Eligio_Uicab_Jatswooj_4,je'elo le túun le óotsil máako'obo leti'obe ko...,03_Eligio_Uicab_Jatswooj,spk_021
81,03_Eligio_Uicab_Jatswooj_5,yúuntun le yúuntune le ku pi'ik'tiko'obe ku ts...,03_Eligio_Uicab_Jatswooj,spk_021
...,...,...,...,...
359,07_Eligio_Uicab_Tomojchi_46,entonkes leti'obe tu tuklo'ob leti'obe ma tu k...,07_Eligio_Uicab_Tomojchi,spk_021
360,07_Eligio_Uicab_Tomojchi_47,jkíimo'obe es ke ka tu yu'ubo'ob u taal leti l...,07_Eligio_Uicab_Tomojchi,spk_021
361,07_Eligio_Uicab_Tomojchi_48,leti'obe jela'an u yiliko'obe tumen leti'obe y...,07_Eligio_Uicab_Tomojchi,spk_021
362,07_Eligio_Uicab_Tomojchi_49,wi'it'o'ob leti'ob chéen le jaaj u k'axmaj u k...,07_Eligio_Uicab_Tomojchi,spk_021


In [10]:
df_3h["utt_num"] = df_3h.groupby("new_spk_id").cumcount() + 1

df_3h.head(5)

,utt_id,maya,spk_id,new_spk_id,utt_num
0,01_Anatolio_Pech_1,xtakumbil xunáan le tsikbalo u k'aaba'e xta'ak...,01_Anatolio_Pech,spk_019,1
1,01_Anatolio_Pech_2,jaaj ta'akumbil xunáano le xunáano jump'éel ko...,01_Anatolio_Pech,spk_019,2
2,01_Anatolio_Pech_3,u k'aaba'e ya'axche palomeke u ya'ala'al ti u ...,01_Anatolio_Pech,spk_019,3
3,01_Anatolio_Pech_4,káan u yila tu bin tu taal u yáa'biltale ku yo...,01_Anatolio_Pech,spk_019,4
4,01_Anatolio_Pech_5,pero leti'e u tuukul beyo le káan weenek le u ...,01_Anatolio_Pech,spk_019,5


In [11]:
df_3h["new_id"] = df_3h["new_spk_id"] + "_utt_" + df_3h["utt_num"].astype(str).str.zfill(4)

df_3h_clean = df_3h.drop(columns=["spk_id"]).rename(columns={
    "new_id":"utt_id",
    "maya":"maya",
    "new_spk_id":"spk_id",
    "utt_id":"filename",
})

In [12]:
df_3h_clean = df_3h_clean[["utt_id", "maya", "spk_id", "filename"]]
df_3h_clean["utt_id"].value_counts()

utt_id
spk_019_utt_0001    1
spk_019_utt_0002    1
spk_019_utt_0003    1
spk_019_utt_0004    1
spk_019_utt_0005    1
                   ..
spk_032_utt_0223    1
spk_032_utt_0224    1
spk_032_utt_0225    1
spk_032_utt_0226    1
spk_032_utt_0227    1
Name: count, Length: 1227, dtype: int64

In [13]:
df_1h = pd.read_csv(ANNOTATIONS / "1h-raw-data.csv")
df_1h.head(5)

,utt_id,maya,spanish,spk_id,start,end
0,c5EgkTbau2o_0000,baach,chachalaca,spk_001,00:00:22.550,00:00:24.450
1,c5EgkTbau2o_0001,chiich,abuela,spk_001,00:00:26.450,00:00:28.250
2,c5EgkTbau2o_0002,ch'íich',pájaro,spk_001,00:00:29.750,00:00:31.350
3,c5EgkTbau2o_0003,ja',agua,spk_001,00:00:32.450,00:00:33.550
4,c5EgkTbau2o_0004,kool,milpa,spk_001,00:00:35.350,00:00:37.050


In [14]:
# crear nuevo id

df_1h["utt_num"] = df_1h.groupby("spk_id").cumcount() + 1
df_1h["new_id"] = df_1h["spk_id"] + "_utt_" + df_1h["utt_num"].astype(str).str.zfill(4)

df_1h.head(5)

,utt_id,maya,spanish,spk_id,start,end,utt_num,new_id
0,c5EgkTbau2o_0000,baach,chachalaca,spk_001,00:00:22.550,00:00:24.450,1,spk_001_utt_0001
1,c5EgkTbau2o_0001,chiich,abuela,spk_001,00:00:26.450,00:00:28.250,2,spk_001_utt_0002
2,c5EgkTbau2o_0002,ch'íich',pájaro,spk_001,00:00:29.750,00:00:31.350,3,spk_001_utt_0003
3,c5EgkTbau2o_0003,ja',agua,spk_001,00:00:32.450,00:00:33.550,4,spk_001_utt_0004
4,c5EgkTbau2o_0004,kool,milpa,spk_001,00:00:35.350,00:00:37.050,5,spk_001_utt_0005


In [15]:
df_1h_clean = df_1h.rename(columns={
    "new_id":"utt_id",
    "maya":"maya",
    "spk_id":"spk_id",
    "utt_id":"filename",
})

df_1h_clean = df_1h_clean[["utt_id", "maya", "spk_id", "filename"]]

df_1h_clean.head(5)

,utt_id,maya,spk_id,filename
0,spk_001_utt_0001,baach,spk_001,c5EgkTbau2o_0000
1,spk_001_utt_0002,chiich,spk_001,c5EgkTbau2o_0001
2,spk_001_utt_0003,ch'íich',spk_001,c5EgkTbau2o_0002
3,spk_001_utt_0004,ja',spk_001,c5EgkTbau2o_0003
4,spk_001_utt_0005,kool,spk_001,c5EgkTbau2o_0004


In [16]:
df_all = pd.concat([df_1h_clean, df_3h_clean], ignore_index=True)
df_all.head(5)

,utt_id,maya,spk_id,filename
0,spk_001_utt_0001,baach,spk_001,c5EgkTbau2o_0000
1,spk_001_utt_0002,chiich,spk_001,c5EgkTbau2o_0001
2,spk_001_utt_0003,ch'íich',spk_001,c5EgkTbau2o_0002
3,spk_001_utt_0004,ja',spk_001,c5EgkTbau2o_0003
4,spk_001_utt_0005,kool,spk_001,c5EgkTbau2o_0004


In [17]:
df_all[df_all["utt_id"] == "spk_021_utt_0030"]

,utt_id,maya,spk_id,filename
1414,spk_021_utt_0030,ka tu ya'alajo'obe bejla'e ma ki bejla'e ma...,spk_021,03_Eligio_Uicab_Jatswooj_30


In [18]:
df_all.to_csv(MANIFESTS / "dataset.csv", index=False)

In [19]:
df_all

,utt_id,maya,spk_id,filename
0,spk_001_utt_0001,baach,spk_001,c5EgkTbau2o_0000
1,spk_001_utt_0002,chiich,spk_001,c5EgkTbau2o_0001
2,spk_001_utt_0003,ch'íich',spk_001,c5EgkTbau2o_0002
3,spk_001_utt_0004,ja',spk_001,c5EgkTbau2o_0003
4,spk_001_utt_0005,kool,spk_001,c5EgkTbau2o_0004
...,...,...,...,...
2530,spk_032_utt_0223,jumpuul osea ma jumpuuli cada administración p...,spk_032,15_Mario_Chan_223
2531,spk_032_utt_0224,como to'on láaj explicado to'on beeta'an to'on...,spk_032,15_Mario_Chan_224
2532,spk_032_utt_0225,le ku xkáakpachtiko'obe ma tu pak'o'obi ma tu ...,spk_032,15_Mario_Chan_225
2533,spk_032_utt_0226,teen xan man ti'ob weye camionadasil arroz kin...,spk_032,15_Mario_Chan_226
